## Import

In [0]:
from pyspark.sql import functions as F
import pyspark
import requests
import zipfile

## Config

In [0]:
pasta = "/Volumes/dev/landing/inmet"

INMET = "https://portal.inmet.gov.br/dadoshistoricos" # .zip
SIDRA = "https://apisidra.ibge.gov.br/values/t/5457/n3/41/v/216/p/2023/c782/40124,40122,40127"
IBGE = "https://servicodados.ibge.gov.br/api/v3/malhas/estados/41/metadados"

## Code

##### INMET

In [0]:
ano = 2015
r = requests.get(f"https://portal.inmet.gov.br/uploads/dadoshistoricos/{ano}.zip", timeout=120)

with open(f"{pasta}/{ano}.zip", "wb") as f:
    f.write(r.content)

caminho = f"/Volumes/workspace/landing/inmet/{ano}.zip"

with open(caminho, "wb") as f:
    f.write(r.content)


with zipfile.ZipFile(caminho, 'r') as zip_ref:
    zip_ref.extractall(pasta)


In [0]:
path = "INMET_S_PR_A835_MARINGA_01-01-2015_A_31-12-2015.CSV"
inmet_schema_location = "/Volumes/dev/bronze/_schemas/inmet/"


df_preview = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .option("skipRows", 8)
    .csv(f"/Volumes/dev/landing/inmet/{ano}/{path}")
)


display(df_preview)

# df_raw = (
#     spark.readStream
#     .format("cloudFiles")
#     .option("cloudFiles.format", "csv")
#     .option("header", "true")
#     .option("sep", ";")
#     .option("encoding", "ISO-8859-1")
#     .option("cloudFiles.schemaLocation", inmet_schema_location )
#     .option("skipRows", 8)
#     .load(f"/Volumes/dev/landing/inmet/{ano}")
# )

In [0]:
r_INMET = requests.get(INMET, timeout=30)

try:
    r_INMET = requests.get(INMET, timeout=30)
    print(f"INMET: {r_INMET.status_code}")
except Exception as e:
    print(f"INMET: FALHOU ({type(e).__name__})")

try:
    r_SIDRA = requests.get(SIDRA, timeout=30)
    print(f"SIDRA: {r_SIDRA.status_code}")
except Exception as e:
    print(f"SIDRA: FALHOU ({type(e).__name__})")

try:
    r_IBGE = requests.get(IBGE, timeout=30)
    print(f"IBGE: {r_IBGE.status_code}")
except Exception as e:
    print(f"IBGE: FALHOU ({type(e).__name__})")

# INMET returns HTML, not JSON — cannot use .json() on it
# r_SIDRA = r_SIDRA.json()
# r_IBGE = r_IBGE.json()

teste = r_SIDRA.json()

print(teste)

# # df_inmet = df = spark.createDataFrame(r_INMET)

# df_inmet